<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Step:
    action: Any
    prob: float  # New: the probability that we ended up here
    reward: float
    state: Any
class MDP:
    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> list[Step]:
        raise NotImplementedError

    def is_end(self, state: Any) -> bool:
        raise NotImplementedError
    def discount(self) -> float:
        raise NotImplementedError
class FlakyTramMDP(MDP):
    def __init__(self, num_locs: int, failure_prob: float):
        self.num_locs = num_locs
        self.failure_prob = failure_prob
    def start_state(self) -> Any:
        return 1

    def successors(self, state: Any) -> list[Step]:
        successors = []
        # Walk
        if state + 1 <= self.num_locs:
            successors.append(Step(action="walk", prob=1, reward=-1, state=state + 1))
        # Tram
        if 2 * state <= self.num_locs:
            # Success: move to desired state
            successors.append(Step(action="tram", prob=1 - self.failure_prob, reward=-2, state=2 * state))
            # Failure: stay in the same state
            successors.append(Step(action="tram", prob=self.failure_prob, reward=-2, state=state))
        return successors
    def is_end(self, state: Any) -> bool:
        return state == self.num_locs
    def discount(self) -> float:
        # No discounting for now
        return 1

#### Environment

In [5]:
import numpy as np

mdp = FlakyTramMDP(num_locs=10, failure_prob=0.4)
np.random.seed(1)

#### Agent

In [6]:
def tram_if_possible_policy(mdp: MDP, state: int) -> str:
    """Need the MDP to know the number of locations to make sure we can take the tram."""
    if state * 2 <= mdp.num_locs:
        return "tram"
    else:
        return "walk"

In [9]:
from typing import Callable
Policy = Callable[[Any], Any]

class RLAlgorithm:
    """
    Abstract class for an RL algorithm, which does two things:
    1. Sends actions to the environment
    2. Incorporates feedback from the environment
    """
    def get_action(self, state: Any) -> Any:
        raise NotImplementedError
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        raise NotImplementedError

class StaticAgent(RLAlgorithm):
    def __init__(self, policy: Policy):
        self.policy = policy
    def get_action(self, state: Any) -> Any:
        return self.policy(state)
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        # Setup Reward Feedback Loigic
        pass

In [10]:
from functools import partial

policy = partial(tram_if_possible_policy, mdp)
rl = StaticAgent(policy=policy)
rl.get_action(state=1)
rl.incorporate_feedback(state=1, action="walk", reward=-1, next_state=2, is_end=False)

In [12]:
from functools import partial

policy = partial(tram_if_possible_policy, mdp)
rl = StaticAgent(policy=policy)
action = rl.get_action(state=1)
rl.incorporate_feedback(state=1, action="walk", reward=-1, next_state=2, is_end=False)

print(f"Action taken: {action}")

Action taken: tram
